# Cafe Sales ETL Pipeline — Python & MS SQL Server

## Project Overview

This notebook performs an end-to-end ETL pipeline on a raw cafe
sales dataset sourced from Kaggle.

**Pipeline Flow:** Raw CSV → Data Exploration → Data Cleaning → SQL Server → Power BI

**Tools Used:** Python, Pandas, NumPy, SQLAlchemy, MS SQL Server

**Dataset:** 10,000 cafe transactions | Year: 2025 | 8 columns

### 1. Importing Libraries
Loading the required Python libraries for data manipulation and database connectivity.

In [1]:
import numpy as np
import pandas as pd

### 2. Data Extraction
Loading the raw dirty dataset from the local source. The dataset contains cafe transaction records with known 
quality issues including nulls, ERROR strings, UNKNOWN labels, and incorrect revenue values.

In [2]:
df = pd.read_csv("dirty_cafe_sales_dataset.csv")
df

,Transaction ID,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Order Type,Transaction Date
0,TXN_1961373,Coffee,20,2,4,Credit Card,Takeaway,08/09/2025
1,TXN_4977031,Cake,30,4,12,Cash,In-store,16/05/2025
2,TXN_4271903,Cookie,10,4,ERROR,Credit Card,In-store,19/07/2025
3,TXN_7034554,Salad,50,2,10,UNKNOWN,UNKNOWN,27/04/2025
4,TXN_3160411,Coffee,20,2,4,Digital Wallet,In-store,11/06/2025
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,20,2,4,NaN,UNKNOWN,30/08/2025
9996,TXN_9659401,NaN,NaN,3,3,Digital Wallet,NaN,02/06/2025
9997,TXN_5255387,Coffee,20,4,8,Digital Wallet,NaN,02/03/2025
9998,TXN_7695629,Cookie,NaN,3,3,Digital Wallet,NaN,02/12/2025


### 3. Exploratory Data Analysis (EDA) 
Understanding the structure, data types, missing value counts, and unique values before any transformation.

### Key observations:
- All columns loaded as string (object) dtype
- Payment Method and Order Type have the highest null rates
- Total Spent column contains ERROR strings
- Duplicate check: No duplicates found

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Price Per Unit    9821 non-null   str  
 3   Quantity          9862 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Order Type        6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
df.describe()

,Transaction ID,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Order Type,Transaction Date
count,10000,9667,9821,9862,9827,7421,6735,9841
unique,10000,10,8,7,19,5,4,367
top,TXN_1961373,Juice,30,5,6,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2429,2013,979,2291,3022,159


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isnull().sum()

Transaction ID         0
Item                 333
Price Per Unit       179
Quantity             138
Total Spent          173
Payment Method      2579
Order Type          3265
Transaction Date     159
dtype: int64

### 4. Data Cleaning & Transformation
Addressing all identified data quality issues.

### Issue || Action Taken: 
- Column names || Renamed to snake case
- ERROR / UNKNOWN strings || Replaced with NaN 
- Null item / price rows || Dropped (cannot recalculate revenue) 
- Null payment / service type || Filled with 'Unknown' 
- Incorrect total_amt values || Recalculated as price × quantity 
- Inconsistent date formats || Standardized to YYYY-MM-DD  

In [7]:
df = df.rename(columns=lambda df: df.strip().lower().replace(" ", "_")) \
     .rename(columns={"total_spent": "total_amt", "order_type": "service_type"})
df

,transaction_id,item,price_per_unit,quantity,total_amt,payment_method,service_type,transaction_date
0,TXN_1961373,Coffee,20,2,4,Credit Card,Takeaway,08/09/2025
1,TXN_4977031,Cake,30,4,12,Cash,In-store,16/05/2025
2,TXN_4271903,Cookie,10,4,ERROR,Credit Card,In-store,19/07/2025
3,TXN_7034554,Salad,50,2,10,UNKNOWN,UNKNOWN,27/04/2025
4,TXN_3160411,Coffee,20,2,4,Digital Wallet,In-store,11/06/2025
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,20,2,4,NaN,UNKNOWN,30/08/2025
9996,TXN_9659401,NaN,NaN,3,3,Digital Wallet,NaN,02/06/2025
9997,TXN_5255387,Coffee,20,4,8,Digital Wallet,NaN,02/03/2025
9998,TXN_7695629,Cookie,NaN,3,3,Digital Wallet,NaN,02/12/2025


In [8]:
unique_values = ['item', 'payment_method', 'service_type']

for x in unique_values:
    print(f"\nColumn_name: {x}")
    print(df[x].unique())


Column_name: item
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

Column_name: payment_method
<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan]
Length: 6, dtype: str

Column_name: service_type
<StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str


In [9]:
df.replace(['nan', 'NaN', '', 'ERROR', 'UNKNOWN'], np.nan, inplace=True)

,transaction_id,item,price_per_unit,quantity,total_amt,payment_method,service_type,transaction_date
0,TXN_1961373,Coffee,20,2,4,Credit Card,Takeaway,08/09/2025
1,TXN_4977031,Cake,30,4,12,Cash,In-store,16/05/2025
2,TXN_4271903,Cookie,10,4,NaN,Credit Card,In-store,19/07/2025
3,TXN_7034554,Salad,50,2,10,NaN,NaN,27/04/2025
4,TXN_3160411,Coffee,20,2,4,Digital Wallet,In-store,11/06/2025
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,20,2,4,NaN,NaN,30/08/2025
9996,TXN_9659401,NaN,NaN,3,3,Digital Wallet,NaN,02/06/2025
9997,TXN_5255387,Coffee,20,4,8,Digital Wallet,NaN,02/03/2025
9998,TXN_7695629,Cookie,NaN,3,3,Digital Wallet,NaN,02/12/2025


In [10]:
df.isnull().sum()

transaction_id         0
item                 969
price_per_unit       533
quantity             479
total_amt            502
payment_method      3178
service_type        3961
transaction_date     460
dtype: int64

In [11]:
df.dtypes

transaction_id      str
item                str
price_per_unit      str
quantity            str
total_amt           str
payment_method      str
service_type        str
transaction_date    str
dtype: object

In [12]:
df['transaction_id'] = df['transaction_id'].astype('string')

df['price_per_unit'] = (df['price_per_unit'].astype(float).round(0).astype('Int64'))

df['quantity'] = (df['quantity'].astype('Int64'))

df['total_amt'] = (df['total_amt'].astype(float).round(0).astype('Int64'))

df[['item', 'payment_method', 'service_type']] = df[['item', 'payment_method', 'service_type']].astype('category')

df['transaction_date'] = pd.to_datetime(df['transaction_date'],format='%d/%m/%Y',errors='coerce')

In [13]:
df.dtypes

transaction_id              string
item                      category
price_per_unit               Int64
quantity                     Int64
total_amt                    Int64
payment_method            category
service_type              category
transaction_date    datetime64[us]
dtype: object

In [14]:
df[df['item'].notna() & df['price_per_unit'].isna()]

,transaction_id,item,price_per_unit,quantity,total_amt,payment_method,service_type,transaction_date
56,TXN_3578141,Cake,<NA>,5,15,NaN,Takeaway,2025-06-27
65,TXN_4987129,Sandwich,<NA>,3,<NA>,NaN,In-store,2025-10-20
68,TXN_8427104,Salad,<NA>,2,10,NaN,In-store,2025-10-27
85,TXN_8035512,Tea,<NA>,3,4,Cash,NaN,2025-10-29
104,TXN_7447872,Juice,<NA>,2,6,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...
9893,TXN_3809533,Juice,<NA>,2,<NA>,Digital Wallet,Takeaway,2025-02-02
9924,TXN_5981429,Juice,<NA>,2,6,Digital Wallet,NaN,2025-12-24
9926,TXN_2464706,Cake,<NA>,4,12,Digital Wallet,Takeaway,2025-11-09
9961,TXN_2153100,Tea,<NA>,2,3,Cash,NaN,2025-12-29


In [15]:
df[df['price_per_unit'].notna() & df['item'].isna()]

,transaction_id,item,price_per_unit,quantity,total_amt,payment_method,service_type,transaction_date
6,TXN_4433211,NaN,30,3,9,NaN,Takeaway,2025-10-06
8,TXN_4717867,NaN,30,5,15,NaN,Takeaway,2025-07-28
14,TXN_8915701,NaN,15,2,3,NaN,In-store,2025-03-21
30,TXN_1736287,NaN,20,5,10,Digital Wallet,NaN,2025-06-02
31,TXN_8927252,NaN,10,2,<NA>,Credit Card,NaN,2025-11-06
...,...,...,...,...,...,...,...,...
9946,TXN_8807600,NaN,40,1,4,Cash,Takeaway,2025-09-24
9951,TXN_4122925,NaN,10,4,4,NaN,Takeaway,2025-10-20
9958,TXN_4125474,NaN,50,2,10,Credit Card,In-store,2025-08-02
9981,TXN_4583012,NaN,40,5,20,Digital Wallet,NaN,2025-02-27


In [16]:
df[df['item'].notna() & df['price_per_unit'].isna()].shape[0]

479

In [17]:
df[df['price_per_unit'].notna() & df['item'].isna()].shape[0]

915

In [18]:
item_prices = {
    "Coffee": 20,
    "Cake": 30,
    "Cookie": 10,
    "Salad": 50,
    "Smoothie": 40,
    "Sandwich": 60,
    "Juice": 35,
    "Tea": 15
}

df["price_per_unit"] = df["item"].map(item_prices).combine_first(df["price_per_unit"])

df.isnull().sum()

transaction_id         0
item                 969
price_per_unit        54
quantity             479
total_amt            502
payment_method      3178
service_type        3961
transaction_date     460
dtype: int64

In [19]:
df[df['item'].notna() & df['price_per_unit'].isna()].shape[0]

0

In [20]:
price_of_items = {
     20: "Coffee",
     30: "Cake",
     10: "Cookie",
     50: "Salad",
     40: "Smoothie",
     60: "Sandwich",
     35: "Juice",
     15: "Tea"
}

df["item"] = df["price_per_unit"].map(price_of_items).combine_first(df["item"])

df.isnull().sum()

transaction_id         0
item                  54
price_per_unit        54
quantity             479
total_amt            502
payment_method      3178
service_type        3961
transaction_date     460
dtype: int64

In [21]:
df[df['price_per_unit'].notna() & df['item'].isna()].shape[0]

0

In [22]:
df[df["item"].isna() | df["price_per_unit"].isna()].shape[0]

54

In [23]:
df.insert(5, 'recalculated_total_amt',(df['quantity'] * df['price_per_unit']).round(0).astype('Int64'))
df

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
0,TXN_1961373,Coffee,20,2,4,40,Credit Card,Takeaway,2025-09-08
1,TXN_4977031,Cake,30,4,12,120,Cash,In-store,2025-05-16
2,TXN_4271903,Cookie,10,4,<NA>,40,Credit Card,In-store,2025-07-19
3,TXN_7034554,Salad,50,2,10,100,NaN,NaN,2025-04-27
4,TXN_3160411,Coffee,20,2,4,40,Digital Wallet,In-store,2025-06-11
...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,20,2,4,40,NaN,NaN,2025-08-30
9996,TXN_9659401,NaN,<NA>,3,3,<NA>,Digital Wallet,NaN,2025-06-02
9997,TXN_5255387,Coffee,20,4,8,80,Digital Wallet,NaN,2025-03-02
9998,TXN_7695629,Cookie,10,3,3,30,Digital Wallet,NaN,2025-12-02


In [24]:
df.isnull().sum()

transaction_id               0
item                        54
price_per_unit              54
quantity                   479
total_amt                  502
recalculated_total_amt     530
payment_method            3178
service_type              3961
transaction_date           460
dtype: int64

In [25]:
df[df["quantity"].isna() & df["total_amt"].notna() & df["price_per_unit"].notna()]

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
20,TXN_3522028,Smoothie,40,<NA>,20,<NA>,Cash,In-store,2025-04-04
55,TXN_5522862,Cookie,10,<NA>,2,<NA>,Credit Card,Takeaway,2025-03-19
57,TXN_2080895,Cake,30,<NA>,3,<NA>,Digital Wallet,In-store,2025-04-19
66,TXN_8501819,Juice,35,<NA>,6,<NA>,Cash,NaN,2025-03-30
117,TXN_2148617,Juice,35,<NA>,9,<NA>,Digital Wallet,NaN,2025-01-10
...,...,...,...,...,...,...,...,...,...
9932,TXN_8502079,Tea,15,<NA>,3,<NA>,Cash,NaN,2025-04-20
9935,TXN_9778251,Tea,15,<NA>,6,<NA>,NaN,Takeaway,2025-11-09
9944,TXN_7495283,Cake,30,<NA>,15,<NA>,Credit Card,Takeaway,2025-04-14
9957,TXN_6487003,Coffee,20,<NA>,8,<NA>,Credit Card,Takeaway,2025-11-15


In [26]:
df.isnull().sum()

transaction_id               0
item                        54
price_per_unit              54
quantity                   479
total_amt                  502
recalculated_total_amt     530
payment_method            3178
service_type              3961
transaction_date           460
dtype: int64

In [27]:
df["recalculated_total_amt"] = (df["quantity"] * df["price_per_unit"]).round(0).astype("Int64")

df.isnull().sum()

transaction_id               0
item                        54
price_per_unit              54
quantity                   479
total_amt                  502
recalculated_total_amt     530
payment_method            3178
service_type              3961
transaction_date           460
dtype: int64

In [28]:
null_columns = ["item", "price_per_unit", "payment_method", "service_type", "recalculated_total_amt"]

df[df[null_columns].isna().all(axis=1)]

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
1377,TXN_8396271,NaN,<NA>,2,2,<NA>,NaN,NaN,2025-09-12
1589,TXN_5245399,NaN,<NA>,5,10,<NA>,NaN,NaN,2025-12-25
1786,TXN_1923349,NaN,<NA>,4,6,<NA>,NaN,NaN,2025-07-06
2227,TXN_3200203,NaN,<NA>,2,8,<NA>,NaN,NaN,2025-12-04
2289,TXN_7524977,NaN,<NA>,4,<NA>,<NA>,NaN,NaN,2025-12-09
3013,TXN_1842697,NaN,<NA>,5,15,<NA>,NaN,NaN,2025-10-25
4092,TXN_1840897,NaN,<NA>,1,5,<NA>,NaN,NaN,2025-06-03
7478,TXN_9710982,NaN,<NA>,4,16,<NA>,NaN,NaN,2025-03-03
9425,TXN_2065683,NaN,<NA>,3,6,<NA>,NaN,NaN,2025-09-21


In [29]:
df = df.dropna(subset=null_columns, how="all")
df.isnull().sum()

transaction_id               0
item                        45
price_per_unit              45
quantity                   479
total_amt                  501
recalculated_total_amt     521
payment_method            3169
service_type              3952
transaction_date           460
dtype: int64

In [30]:
null_columns1 = ["item", "price_per_unit", "quantity", "recalculated_total_amt"]

df[df[null_columns1].isna().all(axis=1)]

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
3779,TXN_7376255,NaN,<NA>,<NA>,25,<NA>,NaN,In-store,2025-05-27
7597,TXN_1082717,NaN,<NA>,<NA>,9,<NA>,Digital Wallet,In-store,2025-12-13
9819,TXN_1208561,NaN,<NA>,<NA>,20,<NA>,Credit Card,NaN,2025-08-19


In [31]:
df = df.dropna(subset=null_columns1, how="all")
df.isnull().sum()

transaction_id               0
item                        42
price_per_unit              42
quantity                   476
total_amt                  501
recalculated_total_amt     518
payment_method            3168
service_type              3951
transaction_date           460
dtype: int64

In [32]:
null_columns2 = ["item", "price_per_unit", "transaction_date"]

df[df[null_columns2].isna().all(axis=1)]

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
9382,TXN_4255580,NaN,<NA>,1,3,<NA>,Cash,NaN,NaT
9454,TXN_1637131,NaN,<NA>,4,6,<NA>,Cash,NaN,NaT
9764,TXN_1688292,NaN,<NA>,3,9,<NA>,Credit Card,In-store,NaT


In [33]:
df = df.dropna(subset=null_columns2, how="all")
df.isnull().sum()

transaction_id               0
item                        39
price_per_unit              39
quantity                   476
total_amt                  501
recalculated_total_amt     515
payment_method            3168
service_type              3949
transaction_date           457
dtype: int64

In [34]:
null_columns3 = ["item", "price_per_unit"]

df[df[null_columns3].isna().any(axis=1)]

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
118,TXN_4633784,NaN,<NA>,5,15,<NA>,NaN,In-store,2025-02-06
151,TXN_4031509,NaN,<NA>,4,16,<NA>,Credit Card,Takeaway,2025-01-04
289,TXN_3495950,NaN,<NA>,4,6,<NA>,Credit Card,In-store,2025-02-19
334,TXN_2523298,NaN,<NA>,4,6,<NA>,NaN,In-store,2025-03-25
550,TXN_4186681,NaN,<NA>,4,6,<NA>,Digital Wallet,NaN,2025-05-24
750,TXN_5787508,NaN,<NA>,3,9,<NA>,Credit Card,Takeaway,2025-07-23
818,TXN_7940202,NaN,<NA>,1,4,<NA>,Digital Wallet,NaN,2025-07-23
1154,TXN_2473090,NaN,<NA>,2,3,<NA>,Credit Card,In-store,2025-03-03
1337,TXN_5031214,NaN,<NA>,5,5,<NA>,NaN,Takeaway,2025-07-29
1761,TXN_3611851,NaN,<NA>,4,<NA>,<NA>,Credit Card,NaN,2025-02-09


In [35]:
df = df.dropna(subset=null_columns3, how="any")
df.isnull().sum()

transaction_id               0
item                         0
price_per_unit               0
quantity                   476
total_amt                  499
recalculated_total_amt     476
payment_method            3157
service_type              3936
transaction_date           457
dtype: int64

In [36]:
for x in ["payment_method", "service_type",]:
    df[x] = df[x].cat.add_categories("Unknown")
    df[x] = df[x].fillna("Unknown")

# Filling with 0 to preserve records; recalculated_total_amt is the trusted revenue column
columns = ["quantity", "total_amt", "recalculated_total_amt"]
df[columns] = df[columns].fillna(0)

df.isnull().sum()

transaction_id              0
item                        0
price_per_unit              0
quantity                    0
total_amt                   0
recalculated_total_amt      0
payment_method              0
service_type                0
transaction_date          457
dtype: int64

In [37]:
df.to_excel("cleaned_cafe_sales_dataset.xlsx", index=False)

### 5. Loading to MS SQL Server
- Connecting to a local SQL Server instance using SQLAlchemy and loading the cleaned DataFrame into the 'cafe' table inside the 'cafe_data' database.
- The table is replaced on each run to ensure fresh data.

In [38]:
from sqlalchemy import create_engine

server   = r'YOUR_SERVER_NAME'
database = 'cafe_data'
username = r'YOUR_USERNAME'
password = 'YOUR_PASSWORD'
driver   = 'ODBC Driver 18 for SQL Server' 
Trusted_Connection = 'yes'

connection_string = f"mssql+pyodbc://@{server}/{database}?trusted_connection=yes&driver={driver.replace(' ', '+')}& Encrypt=no"

engine = create_engine(connection_string)

try:
    conn = engine.connect()
    print("Connection successful!")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

df.to_sql(
    name = 'cafe',      
    con = engine,
    if_exists = 'replace', 
    index = False
)
print("Data successfully loaded!")

Connection successful!
Data successfully loaded!


In [39]:
query = "SELECT * FROM cafe"
df = pd.read_sql(query, connection_string)
df

,transaction_id,item,price_per_unit,quantity,total_amt,recalculated_total_amt,payment_method,service_type,transaction_date
0,TXN_1961373,Coffee,20,2,4,40,Credit Card,Takeaway,2025-09-08
1,TXN_4977031,Cake,30,4,12,120,Cash,In-store,2025-05-16
2,TXN_4271903,Cookie,10,4,0,40,Credit Card,In-store,2025-07-19
3,TXN_7034554,Salad,50,2,10,100,Unknown,Unknown,2025-04-27
4,TXN_3160411,Coffee,20,2,4,40,Digital Wallet,In-store,2025-06-11
...,...,...,...,...,...,...,...,...,...
9941,TXN_7851634,Smoothie,40,4,16,160,Unknown,Unknown,2025-01-08
9942,TXN_7672686,Coffee,20,2,4,40,Unknown,Unknown,2025-08-30
9943,TXN_5255387,Coffee,20,4,8,80,Digital Wallet,Unknown,2025-03-02
9944,TXN_7695629,Cookie,10,3,3,30,Digital Wallet,Unknown,2025-12-02


### 6. Verification
Reading the data back from SQL Server to confirm the pipeline loaded successfully and all 9 columns are intact.

**Final cleaned dataset:** 9,946 rows × 9 columns

**Records dropped during cleaning:** 54 rows

**New column added:** recalculated_total_amt (price × quantity)